# Render a Claude Artifact on Desktop + Mobile and Build a 5-minute Video

**How to use:**
1. Click **Runtime** -> **Run all** (or press `Ctrl/Cmd + F9`).
2. Wait ~5-7 minutes.
3. The final MP4 (`final_video.mp4`) downloads automatically at the end. It is also visible in the file panel on the left (folder icon).

Edit the `URL` variable below to point at any Claude public artifact (or any public web page).

In [ ]:
URL = "https://claude.ai/public/artifacts/9fb88f95-580e-487b-ad57-d6ce0e3a0be5"
DESKTOP_VIEWPORT = {"width": 1920, "height": 1080}
MOBILE_VIEWPORT  = {"width": 390,  "height": 844}
DESKTOP_SECONDS = 140   # ~2:20 of desktop footage
MOBILE_SECONDS  = 140   # ~2:20 of mobile footage
# Plus ~3s title cards x2 = ~5 minutes total

## 1. Install Playwright + Chromium + ffmpeg

In [ ]:
%%capture
!pip install --quiet playwright
!playwright install chromium
!apt-get install -y ffmpeg

## 2. Render the page and record video at both viewports

In [ ]:
import asyncio, os, shutil, glob
from playwright.async_api import async_playwright

shutil.rmtree('recordings', ignore_errors=True)
os.makedirs('recordings/desktop', exist_ok=True)
os.makedirs('recordings/mobile', exist_ok=True)

async def record(viewport, out_dir, duration_s, is_mobile=False, label='view'):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=['--no-sandbox','--disable-dev-shm-usage'])
        context = await browser.new_context(
            viewport=viewport,
            record_video_dir=out_dir,
            record_video_size=viewport,
            is_mobile=is_mobile,
            user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36' if not is_mobile else 'Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Mobile/15E148 Safari/604.1',
        )
        page = await context.new_page()
        print(f'[{label}] navigating to {URL}')
        try:
            await page.goto(URL, wait_until='networkidle', timeout=60000)
        except Exception as e:
            print(f'[{label}] networkidle timeout, falling back: {e}')
            await page.goto(URL, wait_until='load', timeout=60000)
        await page.wait_for_timeout(6000)  # give artifact JS time to render
        # Slow scroll down then back up across `duration_s` seconds.
        half = duration_s // 2
        for _ in range(half):
            await page.evaluate('window.scrollBy(0, 40)')
            await page.wait_for_timeout(1000)
        for _ in range(half):
            await page.evaluate('window.scrollBy(0, -40)')
            await page.wait_for_timeout(1000)
        await page.wait_for_timeout(2000)
        await context.close()
        await browser.close()
        vids = glob.glob(f'{out_dir}/*.webm')
        print(f'[{label}] recorded {vids}')

await record(DESKTOP_VIEWPORT, 'recordings/desktop', DESKTOP_SECONDS, False, 'desktop')
await record(MOBILE_VIEWPORT,  'recordings/mobile',  MOBILE_SECONDS,  True,  'mobile')

## 3. Stitch into a single 5-minute MP4 with title cards

In [ ]:
import glob, subprocess, os

def sh(cmd):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

desktop_webm = sorted(glob.glob('recordings/desktop/*.webm'))[0]
mobile_webm  = sorted(glob.glob('recordings/mobile/*.webm'))[0]

# Re-encode both clips into 1280x720 H.264 with letterboxing so they share params.
sh(['ffmpeg','-y','-i', desktop_webm,
    '-vf','scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black,setsar=1,fps=30',
    '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p','desktop.mp4'])

sh(['ffmpeg','-y','-i', mobile_webm,
    '-vf','scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black,setsar=1,fps=30',
    '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p','mobile.mp4'])

# Title cards (3s each).
for fname, text in [('title_desktop.mp4','Desktop View  -  1920 x 1080'),
                    ('title_mobile.mp4','Mobile View  -  390 x 844')]:
    sh(['ffmpeg','-y','-f','lavfi','-i','color=c=black:s=1280x720:d=3:r=30',
        '-vf', f"drawtext=text='{text}':fontcolor=white:fontsize=56:x=(w-text_w)/2:y=(h-text_h)/2",
        '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p', fname])

with open('concat.txt','w') as f:
    for clip in ['title_desktop.mp4','desktop.mp4','title_mobile.mp4','mobile.mp4']:
        f.write(f"file '{clip}'\n")

sh(['ffmpeg','-y','-f','concat','-safe','0','-i','concat.txt','-c','copy','final_video.mp4'])
print('Final video:', os.path.abspath('final_video.mp4'),
      'size MB:', round(os.path.getsize('final_video.mp4')/1e6, 2))

## 4. Preview + auto-download

In [ ]:
from IPython.display import Video, display
display(Video('final_video.mp4', embed=True, width=720))
try:
    from google.colab import files
    files.download('final_video.mp4')
except Exception as e:
    print('Auto-download not available; open the Files panel on the left and download final_video.mp4 manually.')